# 03 — Modelagem supervisionada e validação

**Tech Challenge Fase 3 — FIAP Pós-Tech IA Scientist**

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # permite `import src` a partir de notebooks/
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src import config as C
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)

## Protocolo

1. Split estratificado 80/20 — o teste é usado **uma única vez**, no final.
2. CV estratificado 5-fold no treino: baselines (Dummy, LogReg, RF, HistGradientBoosting).
3. `RandomizedSearchCV` (scoring = average precision) sobre o pipeline completo.
4. Threshold escolhido em predições *out-of-fold* do treino, exigindo recall da classe risco ≥ 0,85.
5. Avaliação no teste, calibração isotônica, persistência em `models/`.

Tudo isso está encapsulado em `src/modeling/train.py`; aqui executamos e inspecionamos.

In [ ]:
from src.modeling import train
train.main(["--recall-minimo", "0.85"])   # use ["--rapido"] para um smoke test

In [ ]:
import json
meta = json.loads((C.MODELS_DIR / "metadata.json").read_text())
pd.read_csv(C.REPORTS_DIR / "cv_baselines.csv")

In [ ]:
pd.DataFrame({"threshold escolhido": meta["teste_threshold_escolhido"], "threshold 0.5": meta["teste_threshold_0.5"]}).T

## Controle de overfitting: gap treino × CV por modelo

In [ ]:
pd.DataFrame(meta["busca"]).T

In [ ]:
from IPython.display import Image; Image(str(C.IMAGES_DIR / "09_resultados_modelos.png"))

## Variante estrutural (sem bloco histórico) — quanto os fatores estruturais sozinhos explicam?

In [ ]:
train.main(["--sem-historico"])
json.loads((C.MODELS_DIR / "metadata_sem_historico.json").read_text())["teste_threshold_escolhido"]

## Interpretação (preencher)

- Modelo escolhido e por quê (PR-AUC no CV, estabilidade, gap treino/CV).
- Custo do threshold: quantos falsos alarmes por município em risco capturado.
- Diferença entre o modelo completo e a variante estrutural.